# Imputación ampliada: diez metabolitos

Selecciona Ishii o MMC2 y ejecuta todas las celdas. Este notebook **lee los resultados ya calculados**, sin entrenar ni exportar archivos.

El análisis y sus límites están en `docs/METODOLOGIA_AMPLIADA.md`. La selección de parámetros usa validación interna; las tablas de precisión usan exclusivamente evaluación externa.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

DATASET = "ishii"  # ishii o mmc2
METABOLITO = "fum[c]"
METODO = "Iterative"  # Mediana, KNN o Iterative
ROOT = Path.cwd()
RES = ROOT / "resultados_ampliados" / DATASET
if not RES.is_dir():
    raise FileNotFoundError("Abre el notebook desde la carpeta del proyecto.")

## 1. Datos y referencias
Los diez metabolitos se evalúan. Los valores medidos permanecen intactos; ATP, ADP y AMP están completos en Ishii.

In [ ]:
original = pd.read_csv(RES / f"matriz_{DATASET}_mM.csv", index_col=0)
display(pd.DataFrame({"medidos": original.notna().sum(), "faltantes": original.isna().sum()}))
display(pd.read_csv(RES / "homologacion.csv"))

## 2. Selección de parámetros
Una configuración elegida por método y partición. El menor error interno se usa para elegir; no es el error final del método.

In [ ]:
particiones = json.loads((RES / "particiones.json").read_text())
seleccion = [{"particion": p["fold"], "metodo": m, **c} for p in particiones for m, c in p["chosen"].items()]
display(pd.DataFrame(seleccion))
display(pd.read_csv(RES / "hiperparametros.csv"))

## 3. Evaluación externa
Cada punto corresponde a una medición ocultada. El macro MAE_R da el mismo peso a cada metabolito, después de normalizar por su referencia. Consulta también los errores en mM.

In [ ]:
pred = pd.read_csv(RES / "predicciones.csv")
metrics = pd.read_csv(RES / "metricas.csv")
display(pd.read_csv(RES / "comparacion_global.csv"))
display(metrics[metrics.metabolite == METABOLITO])
display(Image(filename=str(RES / "figuras/comparacion_metodos.png")))
display(pd.read_csv(RES / "diagnostico_ajustes.csv"))

## 4. Incertidumbre contrastada con datos
Iterative tiene 30 imputaciones posteriores: SD y cuantiles. Para los tres métodos hay rangos empíricos de residuos internos cuando existen al menos cinco muestras. Ninguno garantiza por sí solo una cobertura del 95 %.

Los gráficos de fumarato y ATP muestran los intervalos posteriores frente a las mediciones ocultadas. La estimación puntual principal es determinista; la media posterior se identifica por separado.

In [ ]:
intervalos = pd.read_csv(RES / "intervalos_evaluacion.csv")
display(intervalos[intervalos.metabolite == METABOLITO])
display(pred[(pred.metabolite == METABOLITO) & (pred.method == "Iterative")][["sample", "truth_mM", "estimate_mM", "posterior_mean_mM", "posterior_SD_mM", "posterior_low_mM", "posterior_high_mM"]])
display(Image(filename=str(RES / "figuras/intervalos_fum.png")))

## 5. Candidatos reales
No conocemos su valor verdadero. Un intervalo estrecho no demuestra exactitud. Las estimaciones se obtienen excluyendo el bloque de grupos de la muestra.

In [ ]:
candidatos = pd.read_csv(RES / "candidatos.csv")
display(candidatos[(candidatos.metabolite == METABOLITO) & (candidatos.method == METODO)])

## 6. Distribuciones
Antes y después comparten las mediciones, por eso no se aplican pruebas de muestras independientes. Los histogramas y estadísticos describen cambios, no validan la imputación. Las pruebas sobre pares medido–estimado son exploratorias y se corrigen por comparaciones múltiples.

In [ ]:
stats = pd.read_csv(RES / "distribuciones.csv")
display(stats[(stats.metabolite == METABOLITO)])
display(Image(filename=str(RES / "figuras" / f"distribuciones_{METODO}.png")))
pruebas = pd.read_csv(RES / "distribuciones_validacion_test.csv")
display(pruebas[pruebas.metabolite == METABOLITO])

## 7. Conclusión y límites
No interpretar p > 0,05 como equivalencia. MMC2 tiene solo ocho condiciones; sus conclusiones y cuantiles son especialmente inestables.

In [ ]:
display(Markdown((RES / "CONCLUSIONES.md").read_text(encoding="utf8")))